In [ ]:
import os
base_path = os.path.dirname(os.path.abspath(""))
base_path = base_path + "/headers/"
print(base_path)
header_path = base_path + 'header_data_treatment.ipynb'
%run $header_path

C:\Users\raphael.wargon\Documents\GitHub\sorting_univ\script/headers/
initalizing spark nlp session w gpu


In [ ]:
works_au_af = (spark.read.format('parquet').load('file:\\' + main_path_openalex + 'works_au_af.parquet')
               .filter( func.col('publication_year').between(1950,2020) )
              )

In [ ]:
df_authors = (spark.read.format('parquet').load('file:\\' + main_path_openalex + 'authors.parquet')
              )

In [ ]:
journals_ranking = (spark.read.format('parquet').load('file:\\' + main_path_openalex + 'journals_ranking_year.parquet')
                    .withColumn('source_id', func.regexp_replace(func.col('source_id'), 'https://openalex.org/', ''))
                   )
journals_ranking.show()

In [ ]:
#df_au_scanR = spark.read.format('parquet').load('file:\\' + scanR_path + '\\authors.parquet')

In [ ]:
papers_textual_metrics = spark.read.csv('file:///' + databases_path + '/papers_textual_metrics.csv', header = True) 
papers_textual_metrics = papers_textual_metrics.withColumnRenamed('paperID', 'work_id').drop(*['n_words','n_phrases', 'has_abstract'])

In [ ]:
papers_textual_metrics.show()

# Build set of French and UK authors

In [ ]:
inst = spark.read.parquet('file:\\' + int_save_path + 'inst_fr.parquet')

In [ ]:
inst_UK = spark.read.parquet('file:\\' + int_save_path + 'inst_UK.parquet')

In [10]:
inst.filter(func.col('inst_id').contains('I4210115196')).show(truncate  = False)

+-----------+----------------------------------------------------+--------+---------+---------------------------+-------------------------------+----------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+-------------------------------------+----+----+-------------------+----------+-------+---------+---------------+-----------+--------------+
|inst_id    |name                                                |type    |city     |homepage_url               |topic_share                    |main_topic      |parent                                                                                                                                                                                      |uni_pub|parent_id                            |cnrs|idex|total_idex_dotation|type_fr   |secteur|acces_rce|date_first_idex|fusion_date|merged_inst_id|


In [10]:
inst.filter(func.col('inst_id')=="I118618916").show()

+----------+--------------------+---------+--------------------+--------------------+--------------------+---------------+--------------------+-------+----+----+-------------------+-------------------+-------+---------+---------------+-----------+--------------+
|   inst_id|                name|     type|                city|        homepage_url|         topic_share|     main_topic|              parent|uni_pub|cnrs|idex|total_idex_dotation|            type_fr|secteur|acces_rce|date_first_idex|fusion_date|merged_inst_id|
+----------+--------------------+---------+--------------------+--------------------+--------------------+---------------+--------------------+-------+----+----+-------------------+-------------------+-------+---------+---------------+-----------+--------------+
|I118618916|Université Pierre...|education|Saint-Martin-d'Hères|http://www.upmf-g...|[{Social Sciences...|Social Sciences|[{I899635006, Uni...|      1|   0|NULL|               NULL|Grand établissement| public|  

In [11]:
set_french_UK_authors = (works_au_af
                      .filter(func.col('publication_year').between(1997,2020))
                     # .withColumn('inst_id', func.when(func.col('inst_id')=='I118618916',  "I899635006")
                     #                            .when(func.col('inst_id')=="I36085230",  "I899635006")
                     #                            .when(func.col('inst_id')=="I177483745",  "I899635006")
                     #                            .when(func.col('inst_id')=="I4210103002", "I198244214")
                     #                            .when(func.col('inst_id')=="I4210143836", "I198244214")
                     #                            .when(func.col('inst_id')=="I184646667",  "I39804081")
                     #                            .when(func.col('inst_id')=="I102197404",  "I277688954")
                     #                            .when(func.col('inst_id')=='I7171862',  "I2279609970")
                     #                            .when(func.col('inst_id')=="I59807433",  "I2279609970")
                     #                            .when(func.col('inst_id')=="I2800379142", "I4210154111") 
                     #                            .when(func.col('inst_id')=="I3123023596","I56067802")
                     #                            .when(func.col('inst_id')=="I4210095130","I208215962")
                     #                            .otherwise(func.col('inst_id')))
                      .join(
                          inst.select('inst_id', func.lit(1).alias('in_inst_dataset'), func.lit('FR').alias('inst_dataset'))
                          .unionAll(inst_UK.select('inst_id', func.lit(1).alias('in_inst_dataset'), func.lit('UK').alias('inst_dataset')))
                          , on =['inst_id'], how = 'left')
                      .fillna(0, subset = ['in_inst_dataset'])
                      #.filter( (func.col('country')=='FR') | (func.col('in_inst_dataset')==1))
                      .groupBy('author_id')
                      .agg(func.max('publication_year').alias('last_year'),
                           func.min('publication_year').alias('entry_year'),
                           func.countDistinct('work_id').alias('n_pub_in_ds'),
                           func.countDistinct('publication_year').alias('n_y_in_ds')
                          )
                      .filter( ~(func.col('last_year')==func.col('entry_year'))
                               & (func.col('n_y_in_ds')>2)
                               & (func.col('n_pub_in_ds')>2)
                             )
                     )
#set_french_authors.cache()
#set_french_authors.count()

In [13]:
#(set_french_authors.filter(func.col('author_id')=="A5084984675")
#                      .join(works_au_af, on = ['author_id'], how = 'left')).show()

In [14]:
#set_french_authors.show()

In [15]:
set_french_UK_authors.write.mode('overwrite').parquet('file:\\' + int_save_path + 'set_french_UK_authors.parquet')

# Load set of French authors

In [ ]:
set_french_UK_authors = spark.read.parquet('file:\\' + int_save_path + 'set_french_UK_authors.parquet')
set_french_UK_authors.count()

In [ ]:
set_french_UK_authors.show()

In [ ]:
works_au_af.show()


In [ ]:
works_au_af_subset = (set_french_UK_authors
                      .join(works_au_af, on = ['author_id'], how = 'left')
                     )
#works_au_af_subset.cache()#
#works_au_af_subset.show()

In [ ]:
works_au_af_subset = (works_au_af_subset
                      .withColumnRenamed('type', 'work_type')
                 # .withColumn('inst_id', func.expr(expr_inst_id))
                 # .join(inst.drop('country').distinct(), on = ['inst_id'])
                  .join(inst.distinct()
                        .unionAll(inst_UK.select(*inst_UK.columns,
                                                 *[func.lit(None).alias(col) for col in inst.columns if col not in inst_UK.columns])
                                  .select(inst.columns) .distinct()), on = ['inst_id'])
                     )
#works_au_af_subset.cache()
#works_au_af_subset.show()

In [18]:
works_au_af_subset.distinct().write.mode('overwrite').parquet('file:\\' + int_save_path + 'works_au_af_subset.parquet') 

In [19]:
works_au_af_subset = spark.read.parquet('file:\\' + int_save_path + 'works_au_af_subset.parquet')

In [26]:
works_au_af_subset.count()

36824502

In [20]:
works_au_af_subset.show()

+-----------+-----------+---------+----------+-----------+---------+-----------+-----------+----------------+---------+------------+---------------+-------+---------+--------------------+--------+----------------+--------------------+-----------------------+--------------------+--------------------+--------------------+---------+------------+--------------------+--------------------+--------------------+--------------------+-------+--------------------+----+----+-------------------+----------+-------+---------+---------------+-----------+--------------+
|    inst_id|  author_id|last_year|entry_year|n_pub_in_ds|n_y_in_ds|    work_id|  source_id|publication_year|citations|   work_type|author_position|country|inst_type|               title|language|         version|       primary_topic|raw_affiliation_strings|             funders|              awards|                name|     type|        city|        homepage_url|         topic_share|          main_topic|              parent|uni_pub|    

# Build author-level characteristics


In [10]:
au_level_char = (works_au_af_subset
                                       .join(inst.select('inst_id', func.lit(1).alias('in_inst_dataset')), on =['inst_id'], how = 'left')
                      .fillna(0, subset = ['in_inst_dataset'])
              .groupBy('author_id','publication_year').agg(func.max((func.col('in_inst_dataset')==1).cast('int')).alias('in_FR'),
                                                          func.max((func.col('in_inst_dataset')==0).cast('int')).alias('abroad'))
              .groupBy('author_id').agg(func.min('publication_year').alias('entry_year'),
                                        func.max('publication_year').alias('last_year'),
                                        func.sum('in_FR').alias('all_y_in_sample'), 
                                        func.sum('abroad').alias('years_abroad'))
                )
au_level_char.cache()
au_level_char.show()

+-----------+----------+---------+---------------+------------+
|  author_id|entry_year|last_year|all_y_in_sample|years_abroad|
+-----------+----------+---------+---------------+------------+
|A5002082143|      1999|     2020|              0|           8|
|A5002496378|      1968|     2010|              0|          29|
|A5002715722|      1988|     2020|             32|           3|
|A5003128416|      2019|     2020|              2|           0|
|A5005282236|      2015|     2020|              0|           6|
|A5005467138|      2011|     2017|              0|           5|
|A5005560517|      2008|     2012|              0|           4|
|A5006541973|      1983|     2020|              0|          14|
|A5007336038|      2009|     2020|              0|           5|
|A5007411109|      1968|     2020|             42|           2|
|A5008572050|      2008|     2020|              0|           9|
|A5008682618|      1968|     1982|              2|           1|
|A5008955924|      2015|     2019|      

In [11]:
au_level_char = (au_level_char
                 .join(works_au_af_subset.groupBy('author_id', func.col('publication_year').alias('entry_year'))
                       .agg(func.concat_ws(',', func.array_sort(func.array_compact(func.collect_set('country'))))
                            .alias('entry_country')),
                       on = ['author_id', 'entry_year'], how = 'left')
                )

In [12]:
au_level_char.filter(func.col('author_id')=="A5084984675").show()

+-----------+----------+---------+---------------+------------+-------------+
|  author_id|entry_year|last_year|all_y_in_sample|years_abroad|entry_country|
+-----------+----------+---------+---------------+------------+-------------+
|A5084984675|      1985|     2020|             28|          30|           FR|
+-----------+----------+---------+---------------+------------+-------------+



In [13]:
gendersort_path =  os.path.dirname(os.path.abspath("")) + "\\additional_material\\Wiki_Gendersort.py"
%run $gendersort_path

In [14]:
udf_wiki_gendersort = func.udf(wiki_gendersort().assign,StringType())
au_fields = (set_french_UK_authors
             .join(df_authors
                  .select(func.regexp_replace(func.col('id'), 'https://openalex.org/', '').alias('author_id'), func.col('display_name').alias('name'),
                    func.explode(func.col('topic_share')).alias('topic_share'))
                   , on = ['author_id'], how = 'inner')
             .withColumn('domain', func.col('topic_share.domain.id')) 
             .withColumn('max_topic', (func.col('topic_share.value') ==
                                                   func.max(func.col('topic_share.value'))
                                          .over(Window.partitionBy('author_id'))).cast('int'))
             .withColumn('topic', func.array_compact(func.collect_set(func.when(func.col('max_topic')==1, func.col('topic_share.id')))
                                                        .over(Window.partitionBy('author_id')) ) )
             .withColumn('subfield_value', func.sum(func.col('topic_share.value'))
                         .over(Window.partitionBy('author_id', 'topic_share.subfield.id')))
             .withColumn('max_subfield', func.max(func.col('subfield_value')).over(Window.partitionBy("author_id")))
             .withColumn('subfield', func.array_compact(func.collect_set(func.when(func.col('subfield_value')>=0.9*func.col('max_subfield'), func.col('topic_share.subfield.id')))
                                                        .over(Window.partitionBy('author_id')) ) )
            
             
             .groupBy('author_id','name', 'domain', 'topic', 'subfield',
                      func.col('topic_share.field.id').alias('field')
                     )
             .agg(func.sum("topic_share.value").alias('value'))
             .withColumn('max_value', func.max(func.col('value')).over(Window.partitionBy('author_id')))
             .filter(func.col('value') >= 0.75*func.col('max_value'))
             .groupBy('author_id', 'name','topic', 'subfield')
             .agg(*[ func.array_sort(func.collect_set(func.col(col))).alias(col)
                  for col in ['domain','field']])
             .withColumn('domain',    func.concat_ws(',',func.array_sort(func.transform(func.col('domain'), lambda x: func.regexp_replace(x, 'https://openalex.org/domains/', '')))))
             .withColumn('field',     func.concat_ws(',',func.array_sort(func.transform(func.col("field"), lambda x: func.regexp_replace(x, 'https://openalex.org/fields/', '')))))
             .withColumn('subfield',  func.concat_ws(',',func.array_sort(func.transform(func.col("subfield"), lambda x: func.regexp_replace(x, 'https://openalex.org/subfields/', '')))))
             .withColumn('topic',     func.concat_ws(',',func.array_sort(func.transform(func.col("topic"), lambda x: func.regexp_replace(x, 'https://openalex.org/', '')))))

            .withColumn('author_first_name', func.element_at(func.split(func.col('name'), ' '), 1))
            .withColumn('topic', func.when(func.length(func.col('topic'))<1, func.lit(None)).otherwise(func.col('topic')))
            .withColumn('topic', func.when(func.length(func.col('subfield'))<1, func.lit(None)).otherwise(func.col('subfield')))

            .select('author_id', func.col('name').alias('author_name'),
                    *[func.concat_ws(',', (func.coalesce(func.col(col), func.col(col +'')))).alias(col) for col in ['domain','field','subfield','topic'] ],
                   func.coalesce(func.col('author_first_name'), func.col('author_first_name')).alias('author_first_name'))

            .withColumn('gender', udf_wiki_gendersort(func.col('author_first_name')))
           )
au_fields

DataFrame[author_id: string, author_name: string, domain: string, field: string, subfield: string, topic: string, author_first_name: string, gender: string]

In [15]:
#au_level_char.count()

In [16]:
au_level_char = (au_level_char
                 .filter((func.col('entry_year')<2020) & (func.col('entry_year')>1954))
                 .join(au_fields, on = ['author_id'], how = 'left')
                 .distinct()
                )

In [17]:
au_level_char

DataFrame[author_id: string, entry_year: int, last_year: int, all_y_in_sample: bigint, years_abroad: bigint, entry_country: string, author_name: string, domain: string, field: string, subfield: string, topic: string, author_first_name: string, gender: string]

In [18]:
#au_level_char.show()

In [19]:
au_level_char.write.mode('overwrite').parquet('file:\\' + int_save_path + 'au_level_char.parquet')

In [ ]:
spark.catalog.clearCache()
spark.stop()